# 01 · Data prep
Builds `data/processed/{train,dev,test}/*.parquet` from the raw CMSB, EDOS and EXIST 2025 files and checks the frozen CMSB split.

Each dataset keeps its own tasks, declared under `tasks` in `configs/data_prep.yaml`:
- CMSB: binary (`sexist`).
- EDOS: Task A binary (`sexist`), Task B 4-class (`category`), Task C 11-class (`vector`); B and C are single-label.
- EXIST 2025: Task 1.1 binary (`sexist`), Task 1.3 multi-label over 5 categories (`labels`).

Reproducibility:
- Raw inputs are SHA-256 hashed and checked against `results/data_manifest.json`.
- Every output has a fixed pyarrow schema and a deterministic row order.
- Each output's content hash must match the manifest.

All settings come from `configs/data_prep.yaml`. A missing manifest stops the run. Set `update_manifest: true` there for the first build or for an intended change; the run then prints what changed and rewrites the manifest. Set it back to `false` afterwards.

Outputs: one folder per split (`processed/train/`, `dev/`, `test/`), plus a run record in `OUTPUT_ROOT/{run_name}/` (config, manifest copy, metrics, package list, README). Test files are built with the same pipeline, so test inputs are built exactly like train/dev. Only `99_final_eval` reads them. This notebook prints no test label statistics.

In [1]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
HF_HOME = PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

In [2]:
# Same pins as 00; local runs use the project venv
PINS = {"pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3"}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")

In [3]:
import datetime
import hashlib

import yaml

# Every setting lives in configs/data_prep.yaml; edit it there, not here
CONFIG_PATH = PROJECT_ROOT / "configs" / "data_prep.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text())
KEYS = {
    "dataset", "task", "model", "method", "seed", "update_manifest",
    "languages", "cmsb_split_file", "processed_dir", "manifest", "parquet", "tasks",
}
assert set(CONFIG) == KEYS, f"{CONFIG_PATH.name}: missing {KEYS - set(CONFIG)}, unknown {set(CONFIG) - KEYS}"
assert isinstance(CONFIG["update_manifest"], bool), f"{CONFIG_PATH.name}: update_manifest must be true or false"
assert set(CONFIG["tasks"]) == {"cmsb", "edos", "exist"}, f"{CONFIG_PATH.name}: tasks must cover cmsb, edos, exist"
for ds, tasks in CONFIG["tasks"].items():
    for t in tasks.values():
        assert t["type"] in {"binary", "multiclass", "multilabel"} and ("classes" in t) == (t["type"] != "binary"), f"bad task {ds}: {t}"

CONFIG["config_sha256"] = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
CONFIG["run_name"] = (
    f"{datetime.datetime.now():%Y%m%d-%H%M}_{CONFIG['dataset']}_{CONFIG['task']}"
    f"_{CONFIG['model']}_{CONFIG['method']}_s{CONFIG['seed']}"
)
# A finished run is never overwritten
assert not (OUTPUT_ROOT / CONFIG["run_name"]).exists(), f"{CONFIG['run_name']} exists: wait a minute and rerun"
print(CONFIG["run_name"])

20261019-1557_all_prep_none_dataprep_s42


In [4]:
import random

import numpy as np

# Nothing below is random; seeded so every notebook has the same cell order
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

In [5]:
import hashlib
import html
import io
import json
import platform
import re
import unicodedata
from importlib.metadata import distributions, version

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ENV = {"python": platform.python_version(), **{p: version(p) for p in PINS}}
print(ENV)
off_pin = {p: ENV[p] for p in PINS if ENV[p] != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"
# Full package list for the run record (pip freeze equivalent; works without pip)
PKG_FREEZE = "\n".join(sorted({f"{d.metadata['Name']}=={d.version}" for d in distributions()}, key=str.lower)) + "\n"
# Hash of the notebook as saved on disk: save before running so it matches what ran
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "01_data_prep.ipynb"
NOTEBOOK_SHA = hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest()

# pandas checks for pyarrow once, at import; if pyarrow was installed later, read_parquet breaks
try:
    _buf = io.BytesIO()
    pd.DataFrame({"a": [1]}).to_parquet(_buf)
    pd.read_parquet(io.BytesIO(_buf.getvalue()))
except Exception as e:
    raise RuntimeError("pandas can't use pyarrow in this kernel (imported before pyarrow was installed?): restart the kernel") from e

{'python': '3.12.15', 'pandas': '3.0.6', 'numpy': '2.5.4', 'scikit-learn': '1.9.1', 'pyarrow': '26.0.0', 'pyyaml': '6.0.3'}


## Raw inputs
Hash every file the pipeline reads and stop if any differs from the manifest.

In [6]:
EXIST_DIR = "EXIST/EXIST 2025 Tweets Dataset"
GOLDS = "EXIST/evaluation/golds"
RAW = {
    "cmsb_data": "CMSB/sexism_data.csv",
    "cmsb_annotations": "CMSB/sexism_annotations.csv",
    "cmsb_split": CONFIG["cmsb_split_file"],
    "edos_aggregated": "EDOS/edos_labelled_aggregated.csv",
    "edos_individual": "EDOS/edos_labelled_individual_annotations.csv",
    "exist_train": f"{EXIST_DIR}/training/EXIST2025_training.json",
    "exist_dev": f"{EXIST_DIR}/dev/EXIST2025_dev.json",
    "exist_test": f"{EXIST_DIR}/test/EXIST2025_test_clean.json",
}
for split, stem in [("train", "training"), ("dev", "dev")]:
    for task in ["1_1", "1_2", "1_3"]:
        for kind in ["hard", "soft"]:
            RAW[f"exist_{split}_{task}_{kind}"] = f"{GOLDS}/EXIST2025_{stem}_task{task}_gold_{kind}.json"

# On Colab, read raw files from local disk for speed; outputs still go to Drive
RAW_ROOT = Path("/content/data") if IN_COLAB else DATA_ROOT
if IN_COLAB:
    for p in RAW.values():
        (RAW_ROOT / p).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DATA_ROOT / p, RAW_ROOT / p)


def sha256(path):
    with open(path, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


RAW_SHA = {k: sha256(RAW_ROOT / p) for k, p in RAW.items()}
MANIFEST_PATH = PROJECT_ROOT / CONFIG["manifest"]
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if MANIFEST is None and not CONFIG["update_manifest"]:
    raise FileNotFoundError(f"no manifest at {MANIFEST_PATH}: check PROJECT_ROOT, or set update_manifest: true in {CONFIG_PATH.name} for a first build")
CHECK_MANIFEST = not CONFIG["update_manifest"]

old_raw = {k: v["sha256"] for k, v in MANIFEST["raw"].items()} if MANIFEST else {}
raw_changed = sorted(k for k in RAW_SHA.keys() | old_raw.keys() if RAW_SHA.get(k) != old_raw.get(k))
if MANIFEST and MANIFEST["env"] != ENV:
    print("warning: environment differs from the manifest's:", MANIFEST["env"])
if CHECK_MANIFEST:
    assert not raw_changed, f"raw files differ from {CONFIG['manifest']}: {raw_changed}"
    print("raw files match the manifest")
elif MANIFEST is None:
    print("first build: no manifest yet")
else:
    print("update_manifest is true; raw files changed since the manifest:", raw_changed or "none")

first build: no manifest yet


## Text normalisation
`text` is kept exactly as released. `text_norm` is shared by every model so encoders and LLMs see the same input.

The three datasets mark users and links differently:
- CMSB: `MENTION123` and raw URLs.
- EDOS: `[USER]` and `[URL]`.
- EXIST: raw `@user` handles, raw URLs and HTML entities.

`text_norm` maps all of them to EDOS's `[USER]` / `[URL]`:
- Runs of `[USER]` collapse to one.
- A retweet glued to its mention (`RTMENTION12:`, 2 CMSB posts) is split first, so it becomes `RT [USER]:` like the others.
- URLs stop before trailing punctuation.
- NFC, not NFKC, so stylised characters survive.
- No lowercasing; emoji are kept.

In [7]:
# URL ends before trailing punctuation that is followed by whitespace or the end of the text
URL_RE = re.compile(r"(?:https?://|www\.)\S+?(?=[)\]}>.,!?;:\"']*(?:\s|$))")
# Lookbehind keeps Spanish inclusive forms like "tod@s"; the repeat swallows glued handles "@a@b"
USER_RE = re.compile(r"(?<!\w)@\w+(?:@\w+)*|\bMENTION\d+\b")
USER_RUN_RE = re.compile(r"\[USER\](?:\s*\[USER\])+")
# CMSB has a few retweets glued to the mention ("RTMENTION12:"); split them like "RT MENTION12:"
GLUED_RT_RE = re.compile(r"\bRT(?=MENTION\d+\b)")


def normalise(text):
    """Shared, model-agnostic text cleanup (see the cell above)."""
    text = GLUED_RT_RE.sub("RT ", unicodedata.normalize("NFC", html.unescape(text)))
    text = USER_RUN_RE.sub("[USER]", USER_RE.sub("[USER]", URL_RE.sub("[URL]", text)))
    return " ".join(text.split())


assert normalise("@a @b@c hi (see https://x.co/p?q=1).") == "[USER] hi (see [URL])."
assert normalise("tod@s &amp; MENTION12 www.x.org/a.b") == "tod@s & [USER] [URL]"
assert normalise("RTMENTION3: hi") == normalise("RT MENTION3: hi") == "RT [USER]: hi"

## Schema and writing helpers

In [8]:
def make_schema(extras):
    """Columns shared by every dataset, then that dataset's own task columns."""
    return pa.schema([
        ("id", pa.string()),
        ("text", pa.string()),
        ("text_norm", pa.string()),
        ("lang", pa.string()),
        ("split", pa.string()),
        ("sexist", pa.int64()),
        ("sexist_soft", pa.float64()),
        ("n_annot", pa.int64()),
        ("empty_norm", pa.bool_()),
        ("dup_group", pa.string()),
        ("leak_group", pa.string()),
        *extras,
    ])


def cat_struct(cats, type_):
    """One field per class, in config order."""
    return pa.struct([(c, type_) for c in cats])


def to_table(df, schema):
    # Deterministic row order; NaN becomes null under the explicit schema
    df = df.sort_values(["split", "id"], kind="stable").reset_index(drop=True)
    return pa.Table.from_pandas(df[schema.names], schema=schema, preserve_index=False)


def type_sig(t):
    """Canonical type description; avoids pyarrow's display strings, which can change between versions."""
    if pa.types.is_struct(t):
        return ["struct", [[f.name, type_sig(f.type), f.nullable] for f in t]]
    if pa.types.is_list(t):
        return ["list", type_sig(t.value_type)]
    return str(t)


def content_hash(table):
    """SHA-256 of schema + rows, ignoring file metadata."""
    schema = [[f.name, type_sig(f.type), f.nullable] for f in table.schema]
    blob = json.dumps({"schema": schema, "rows": table.to_pylist()}, sort_keys=True, ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(blob.encode()).hexdigest()


# Pinned hash of a fixed table: fails if the hashing itself ever changes
_t = pa.table({"id": ["a"], "x": [0.5], "l": [["b"]], "s": [{"c": 1.0}]})
assert content_hash(_t) == "ba4e8ba22cb8969590dd31f1f9b06c69e096bdfbb0f930d98169e8a2c4140501", content_hash(_t)


def write_parquet(table, path, sources):
    """Atomic write; skips the file when its content is already identical. Returns True if written."""
    digest = content_hash(table)
    if path.exists() and content_hash(pq.read_table(path)) == digest:
        return False
    # Provenance goes into the file footer; merge so the pandas metadata survives
    meta = {
        "pipeline": "notebooks/01_data_prep.ipynb",
        "run_name": CONFIG["run_name"],
        "sources": {k: RAW_SHA[k] for k in sources},
        "env": ENV,
        "content_sha256": digest,
    }
    table = table.replace_schema_metadata(
        {**(table.schema.metadata or {}), b"heft": json.dumps(meta, sort_keys=True).encode()}
    )
    tmp = path.with_suffix(".parquet.tmp")
    pq.write_table(table, tmp, row_group_size=table.num_rows, **CONFIG["parquet"])
    os.replace(tmp, path)
    return True

## CMSB
Binary task only: `sexist` is the released label, `sexist_soft` the share of the 5 annotators voting sexist.
- `other` posts are unannotated: `annotated` is False and `sexist_soft` and `n_annot` are null.
- These posts are all non-sexist and from a different source, so `annotated == True` is the main setting.
- The source-subset column and the per-annotator content codes are not used, so they are not kept.

In [9]:
cmsb = pd.read_csv(RAW_ROOT / RAW["cmsb_data"])
ann = pd.read_csv(RAW_ROOT / RAW["cmsb_annotations"])
assert len(cmsb) == 13631 and cmsb.id.is_unique, f"CMSB: expected 13,631 unique posts, got {len(cmsb)}"
orphans = sorted(set(ann.id) - set(cmsb.id))
assert orphans == [3345, 4266, 9821, 11501], f"CMSB: unexpected annotation orphans {orphans}"
ann = ann[ann.id.isin(cmsb.id)]
n_annot = ann.groupby("id").size()
assert n_annot.eq(5).all(), "CMSB: every annotated post should have 5 annotations"
# A vote counts as sexist if content is 1-4 or phrasing is 1 (verified rule, data/CLAUDE.md)
sexist_votes = (ann.content.between(1, 4) | ann.phrasing.eq(1)).groupby(ann.id).mean()

cmsb_df = pd.DataFrame({
    "id": cmsb.id.astype(str),
    "text": cmsb.text,
    "text_norm": cmsb.text.map(normalise),
    "lang": "en",
    "sexist": cmsb.sexist.astype("Int64"),
    "sexist_soft": cmsb.id.map(sexist_votes),
    "n_annot": cmsb.id.map(n_annot).astype("Int64"),
    "annotated": cmsb.id.isin(n_annot.index),
    "of_id": pd.Series([None if o == -1 else str(o) for o in cmsb.of_id], dtype=object),
})

### Frozen CMSB split
The script that generated the split wasn't kept, and a fresh `StratifiedGroupKFold` run does not reproduce it. So the file itself is the source of truth: its hash is pinned in the manifest, and the cell below checks the properties it was built to have.

Raw-text grouping holds. After normalisation, though, a few short posts become identical inputs (e.g. only `[USER]` differed) across splits. The split stays frozen; those rows get a `leak_group` id (see *Text audit flags*) so analyses can exclude them.

Sexist rates are printed for train/dev only. Test rows stay in `cmsb_df` and are written to `processed/test/`.

In [10]:
split = pd.read_csv(RAW_ROOT / RAW["cmsb_split"], dtype={"id": str, "split": str}, keep_default_na=False)
assert split.id.is_unique and set(split.id) == set(cmsb_df.id), "CMSB: split ids don't match the data"
cmsb_df["split"] = cmsb_df.id.map(split.set_index("id").split)
sizes = cmsb_df.split.value_counts().to_dict()
assert sizes == {"train": 9543, "dev": 1363, "test": 2725}, f"CMSB: split sizes changed: {sizes}"

# Grouping: an original and its modifications, and duplicate texts, never cross splits
split_of = cmsb_df.set_index("id").split
mods = cmsb_df[cmsb_df.of_id.notna()]
assert (mods.split.values == mods.of_id.map(split_of).values).all(), "CMSB: of_id pair crosses splits"
dup_key = cmsb_df.text.str.casefold().str.split().str.join(" ")
assert cmsb_df.groupby(dup_key).split.nunique().max() == 1, "CMSB: duplicate raw texts cross splits"

# Stratification holds overall and in the annotated-only main setting; test rates are checked, not printed
for name, rows, tol in [("all", cmsb_df, 0.005), ("annotated", cmsb_df[cmsb_df.annotated], 0.01)]:
    rates = rows.groupby("split").sexist.mean()
    assert rates.max() - rates.min() < tol, f"CMSB: {name} sexist rate differs across splits"
    print(f"sexist rate ({name}):", rates[["train", "dev"]].round(4).to_dict())

# The soft-vote rule reproduces the released binary label (>= 3 of 5)
a = cmsb_df.annotated
assert (cmsb_df.sexist_soft[a].ge(0.6).astype("Int64") == cmsb_df.sexist[a]).all(), "CMSB: vote rule mismatch"
assert cmsb_df.sexist[~a].eq(0).all() and a.sum() == 5646, "CMSB: unannotated posts should be non-sexist"

sexist rate (all): {'train': 0.1328, 'dev': 0.1328}
sexist rate (annotated): {'train': 0.3204, 'dev': 0.3198}


## EDOS
- Hard labels come from the aggregated file: Task A `sexist`, Task B `category` (4 classes) and Task C `vector` (11 classes, stored as the official codes `1.1`–`4.2`). B and C are one class per sexist post, null for non-sexist.
- `sexist_soft`, `category_soft`/`category_votes` and `vector_soft`/`vector_votes` come from the 3 individual annotations; each annotator picks at most one class, so a row's shares sum to at most 1.
- The aggregated label is not a plain majority vote, so hard and soft can disagree; the count below covers train/dev only.
- Test rows are built the same way and written to `processed/test/`.

In [11]:
EDOS_TASKS = CONFIG["tasks"]["edos"]
B_CLASSES, C_CLASSES = EDOS_TASKS["task_b"]["classes"], EDOS_TASKS["task_c"]["classes"]
B_NAMES = {
    "1. threats, plans to harm and incitement": "threats",
    "2. derogation": "derogation",
    "3. animosity": "animosity",
    "4. prejudiced discussions": "prejudiced_discussions",
}
assert list(B_NAMES.values()) == B_CLASSES, "EDOS: Task B classes in the config don't match the raw names"


def b_id(v):
    return B_NAMES.get(v, v)


def c_id(v):
    # "2.1 descriptive attacks" -> "2.1"; "none" stays "none"
    return v.split(" ", 1)[0]


agg = pd.read_csv(RAW_ROOT / RAW["edos_aggregated"])
ind = pd.read_csv(RAW_ROOT / RAW["edos_individual"])
sizes = agg.split.value_counts().to_dict()
assert agg.rewire_id.is_unique and sizes == {"train": 14000, "dev": 2000, "test": 4000}, f"EDOS: split sizes {sizes}"
assert ind.groupby("rewire_id").size().eq(3).all() and set(ind.rewire_id) == set(agg.rewire_id), "EDOS: expected 3 annotations per post"
for df, name in [(agg, "aggregated"), (ind, "individual")]:
    b, c = df.label_category.map(b_id), df.label_vector.map(c_id)
    assert set(b) == {*B_CLASSES, "none"} and set(c) == {*C_CLASSES, "none"}, f"EDOS {name}: unexpected class names"
    assert ((df.label_sexist == "sexist") == (b != "none")).all(), f"EDOS {name}: Task A/B disagree"
    # Task C refines Task B: vector 2.x sits under category 2
    assert (c.where(c == "none", c.str[0]) == df.label_category.str[0].where(b != "none", "none")).all(), f"EDOS {name}: Task B/C disagree"


def shares(col, to_id, classes):
    """Per-post annotator votes and shares (out of 3) for one task, in config class order."""
    votes = pd.crosstab(ind.rewire_id, ind[col].map(to_id)).reindex(columns=classes, fill_value=0)
    return (
        {i: row.to_dict() for i, row in votes.div(3).iterrows()},
        {i: {c: int(v) for c, v in row.items()} for i, row in votes.iterrows()},
    )


b_soft, b_votes = shares("label_category", b_id, B_CLASSES)
c_soft, c_votes = shares("label_vector", c_id, C_CLASSES)
sexist_votes = ind.label_sexist.eq("sexist").groupby(ind.rewire_id).mean()

edos_df = pd.DataFrame({
    "id": agg.rewire_id,
    "text": agg.text,
    "text_norm": agg.text.map(normalise),
    "lang": "en",
    "split": agg.split,
    "sexist": agg.label_sexist.eq("sexist").astype("Int64"),
    "sexist_soft": agg.rewire_id.map(sexist_votes),
    "n_annot": pd.Series(3, index=agg.index, dtype="Int64"),
    "category": agg.label_category.map(b_id).where(agg.label_category != "none"),
    "vector": agg.label_vector.map(c_id).where(agg.label_vector != "none"),
    "category_soft": pd.Series([b_soft[i] for i in agg.rewire_id], dtype=object, index=agg.index),
    "category_votes": pd.Series([b_votes[i] for i in agg.rewire_id], dtype=object, index=agg.index),
    "vector_soft": pd.Series([c_soft[i] for i in agg.rewire_id], dtype=object, index=agg.index),
    "vector_votes": pd.Series([c_votes[i] for i in agg.rewire_id], dtype=object, index=agg.index),
})
dev_train = edos_df[edos_df.split != "test"]
conflict = (dev_train.sexist == 1) != (dev_train.sexist_soft >= 0.5)
print(f"EDOS hard label disagrees with the annotator majority on {conflict.sum()} of {len(dev_train)} train/dev posts")

EDOS hard label disagrees with the annotator majority on 772 of 16000 train/dev posts


## EXIST 2025
- Targets come from the official golds.
- `sexist` is the Task 1.1 hard gold; `labels` and `labels_soft` come from Task 1.3, with `NO` dropped (`[]` = not sexist).
- Annotator counts leave out `UNKNOWN` votes, as the golds do:
  - `n_annot` counts the Task 1.1 annotators behind `sexist_soft`. This is the same meaning as in CMSB and EDOS.
  - `labels_n_annot` counts the Task 1.3 annotators behind `labels_votes`. In English train it is lower than `n_annot` for 53 tweets.
- The cell asserts that votes / count equals the official soft gold exactly, for both tasks.
- 1.3 shares can sum to more than 1, because an annotator may pick several categories.
- Tweets with no hard majority get null hard labels. Test has no golds, so its label fields are all null.
- Only `CONFIG["languages"]` is kept; the count checks run on the full release first.

In [12]:
EXIST_CATS = CONFIG["tasks"]["exist"]["task_1_3"]["classes"]
EXIST_INTENT = CONFIG["tasks"]["exist"]["task_1_2"]["classes"]


def load_gold(key):
    return {r["id"]: r["value"] for r in json.loads((RAW_ROOT / RAW[key]).read_text())}


rows = []
for split in ["train", "dev", "test"]:
    tweets = json.loads((RAW_ROOT / RAW[f"exist_{split}"]).read_text())
    gold = {} if split == "test" else {
        f"{t}_{k}": load_gold(f"exist_{split}_{t}_{k}") for t in ["1_1", "1_2", "1_3"] for k in ["hard", "soft"]
    }
    if gold:
        assert set(gold["1_1_soft"]) == set(gold["1_2_soft"]) == set(gold["1_3_soft"]) == set(tweets), f"EXIST {split}: soft golds don't cover all tweets"
        assert all(set(gold[f"{t}_hard"]) <= set(tweets) for t in ["1_1", "1_2", "1_3"]), f"EXIST {split}: unknown gold ids"
        assert set(gold["1_2_hard"].values()) <= set(EXIST_INTENT), f"EXIST {split}: unexpected 1.2 hard values"
        # 1.2 NO means not sexist, so it must agree with 1.1 wherever both have a hard label
        both = gold["1_1_hard"].keys() & gold["1_2_hard"].keys()
        assert all((gold["1_1_hard"][i] == "NO") == (gold["1_2_hard"][i] == "NO") for i in both), f"EXIST {split}: 1.1/1.2 NO disagree"
        # Any other 1.1 value would silently become a null label below
        assert set(gold["1_1_hard"].values()) <= {"YES", "NO"}, f"EXIST {split}: unexpected 1.1 hard values"
    for tid, r in tweets.items():
        assert tid == r["id_EXIST"] and r["split"] == f"{split.upper()}_{r['lang'].upper()}", f"EXIST: bad record {tid}"
        hard13 = gold["1_3_hard"].get(tid) if gold else None
        assert hard13 is None or hard13 == ["NO"] or "NO" not in hard13, f"EXIST: mixed NO label {tid}"
        votes = votes12 = n11 = n12 = n13 = None
        if gold:
            # Per-task annotator counts: n11 backs sexist_soft, n13 backs labels_votes
            anns11 = [a for a in r["labels_task1_1"] if a != "UNKNOWN"]
            n11 = len(anns11)
            assert anns11.count("YES") / n11 == gold["1_1_soft"][tid]["YES"], f"EXIST: 1.1 votes don't match soft gold {tid}"
            # 1.2: one label per annotator, "-" (not sexist) is NO in the golds
            anns12 = ["NO" if a == "-" else a for a in r["labels_task1_2"] if a != "UNKNOWN"]
            n12 = len(anns12)
            votes12 = {c: anns12.count(c) for c in EXIST_INTENT}
            assert all(votes12[c] / n12 == gold["1_2_soft"][tid][c] for c in EXIST_INTENT), f"EXIST: 1.2 votes don't match soft gold {tid}"
            anns = [a for a in r["labels_task1_3"] if "UNKNOWN" not in a]
            n13 = len(anns)
            votes = {c: sum(c in a for a in anns) for c in EXIST_CATS}
            assert all(votes[c] / n13 == gold["1_3_soft"][tid][c] for c in EXIST_CATS), f"EXIST: votes don't match soft gold {tid}"
        rows.append({
            "id": tid,
            "text": r["tweet"],
            "text_norm": normalise(r["tweet"]),
            "lang": r["lang"],
            "split": split,
            "sexist": {"YES": 1, "NO": 0}.get(gold["1_1_hard"].get(tid)) if gold else None,
            "sexist_soft": gold["1_1_soft"][tid]["YES"] if gold else None,
            "n_annot": n11,
            "intention": gold["1_2_hard"].get(tid) if gold else None,
            "intention_soft": {c: gold["1_2_soft"][tid][c] for c in EXIST_INTENT} if gold else None,
            "intention_votes": votes12,
            "intention_n_annot": n12,
            "labels": None if hard13 is None else [c for c in hard13 if c != "NO"],
            "labels_soft": {c: gold["1_3_soft"][tid][c] for c in EXIST_CATS} if gold else None,
            "labels_votes": votes,
            "labels_n_annot": n13,
        })

exist_df = pd.DataFrame(rows)
for col in ["sexist", "n_annot", "intention_n_annot", "labels_n_annot"]:
    exist_df[col] = exist_df[col].astype("Int64")
counts = exist_df.groupby(["split", "lang"]).size().to_dict()
assert counts == {("dev", "en"): 489, ("dev", "es"): 549, ("test", "en"): 978,
                  ("test", "es"): 1098, ("train", "en"): 3260, ("train", "es"): 3660}, f"EXIST: split sizes {counts}"
hard_counts = exist_df.groupby("split").labels.count().to_dict()
assert hard_counts == {"dev": 931, "test": 0, "train": 6050}, f"EXIST: 1.3 hard gold counts {hard_counts}"
hard_counts = exist_df.groupby("split").intention.count().to_dict()
assert hard_counts == {"dev": 841, "test": 0, "train": 5496}, f"EXIST: 1.2 hard gold counts {hard_counts}"
hard_counts = exist_df.groupby("split").sexist.count().to_dict()
assert hard_counts == {"dev": 934, "test": 0, "train": 6064}, f"EXIST: 1.1 hard gold counts {hard_counts}"

exist_df = exist_df[exist_df.lang.isin(CONFIG["languages"])].reset_index(drop=True)
print(exist_df.groupby(["split", "lang"]).size().to_dict())

{('dev', 'en'): 489, ('test', 'en'): 978, ('train', 'en'): 3260}


## Text audit flags
Shared by all three datasets and computed on `text_norm`, the model input. Nothing is removed; each experiment decides what to exclude and records it.
- `empty_norm`: the input is only `[USER]` / `[URL]` placeholders.
- `dup_group`: id shared by rows with identical `text_norm` within a dataset.
- `leak_group`: the same id, set only when that text appears in more than one split.

Counts are per split and use no labels; label conflicts among duplicates are counted on train/dev only.

In [13]:
PLACEHOLDER_ONLY = re.compile(r"(?:\[USER\]|\[URL\]|\s)*")


def add_text_flags(df):
    key = df.text_norm.map(lambda s: hashlib.sha256(s.encode()).hexdigest()[:12])
    by_text = df.groupby("text_norm").split
    return df.assign(
        empty_norm=df.text_norm.map(lambda s: PLACEHOLDER_ONLY.fullmatch(s) is not None),
        dup_group=key.where(by_text.transform("size") > 1),
        leak_group=key.where(by_text.transform("nunique") > 1),
    )


cmsb_df, edos_df, exist_df = (add_text_flags(df) for df in (cmsb_df, edos_df, exist_df))
AUDIT = {}
for name, df in [("cmsb", cmsb_df), ("edos", edos_df), ("exist", exist_df)]:
    counts = df[["empty_norm"]].assign(dup_rows=df.dup_group.notna(), leak_rows=df.leak_group.notna()).groupby(df.split).sum()
    AUDIT[name] = {f"{c}_{s}": int(counts.at[s, c]) for c in counts.columns for s in ["train", "dev", "test"]}
    dev_train = df[(df.split != "test") & df.dup_group.notna()]
    AUDIT[name]["dup_groups_conflicting_sexist_train_dev"] = int(dev_train.groupby("dup_group").sexist.nunique().gt(1).sum())
pd.DataFrame(AUDIT).T

,empty_norm_train,empty_norm_dev,empty_norm_test,dup_rows_train,dup_rows_dev,dup_rows_test,leak_rows_train,leak_rows_dev,leak_rows_test,dup_groups_conflicting_sexist_train_dev
cmsb,67,12,16,543,76,158,108,19,33,8
edos,0,0,0,0,0,0,0,0,0,0
exist,0,0,0,4,0,1,2,0,1,0


## Build tables and check them against the manifest
The task spec in the config is checked against the stored columns and classes. Hashes are checked *before* anything is written, so a mismatch never overwrites good files.

In [14]:
CMSB_SCHEMA = make_schema([("annotated", pa.bool_()), ("of_id", pa.string())])
EDOS_SCHEMA = make_schema([
    ("category", pa.string()),
    ("vector", pa.string()),
    ("category_soft", cat_struct(B_CLASSES, pa.float64())),
    ("category_votes", cat_struct(B_CLASSES, pa.int64())),
    ("vector_soft", cat_struct(C_CLASSES, pa.float64())),
    ("vector_votes", cat_struct(C_CLASSES, pa.int64())),
])
EXIST_SCHEMA = make_schema([
    ("intention", pa.string()),
    ("intention_soft", cat_struct(EXIST_INTENT, pa.float64())),
    ("intention_votes", cat_struct(EXIST_INTENT, pa.int64())),
    ("intention_n_annot", pa.int64()),
    # Child named "element" to match what Parquet's compliant nested type reads back
    ("labels", pa.list_(pa.field("element", pa.string()))),
    ("labels_soft", cat_struct(EXIST_CATS, pa.float64())),
    ("labels_votes", cat_struct(EXIST_CATS, pa.int64())),
    ("labels_n_annot", pa.int64()),
])

# Every configured task must point at a stored column holding only its declared classes
for ds, df, schema in [("cmsb", cmsb_df, CMSB_SCHEMA), ("edos", edos_df, EDOS_SCHEMA), ("exist", exist_df, EXIST_SCHEMA)]:
    for task, t in CONFIG["tasks"][ds].items():
        assert t["column"] in schema.names, f"{ds} {task}: no column {t['column']}"
        vals = df[t["column"]].dropna()
        allowed = {0, 1} if t["type"] == "binary" else set(t["classes"])
        found = set(vals.explode().dropna()) if t["type"] == "multilabel" else set(vals)
        assert found <= allowed, f"{ds} {task}: values outside the config classes: {found - allowed}"

CMSB_SRC = ["cmsb_data", "cmsb_annotations", "cmsb_split"]
EDOS_SRC = ["edos_aggregated", "edos_individual"]
TABLES = {}
for s in ["train", "dev", "test"]:
    TABLES[f"cmsb_{s}"] = (to_table(cmsb_df[cmsb_df.split == s], CMSB_SCHEMA), CMSB_SRC)
    TABLES[f"edos_{s}"] = (to_table(edos_df[edos_df.split == s], EDOS_SCHEMA), EDOS_SRC)
    src = [f"exist_{s}"] + ([] if s == "test" else [k for k in RAW if k.startswith(f"exist_{s}_")])
    TABLES[f"exist_{s}"] = (to_table(exist_df[exist_df.split == s], EXIST_SCHEMA), src)

PROCESSED = {name: {"rows": t.num_rows, "content_sha256": content_hash(t)} for name, (t, _) in TABLES.items()}
old_processed = MANIFEST["processed"] if MANIFEST else {}
processed_changed = sorted(k for k in PROCESSED.keys() | old_processed.keys() if PROCESSED.get(k) != old_processed.get(k))
if CHECK_MANIFEST:
    assert MANIFEST["languages"] == CONFIG["languages"], f"manifest was built for {MANIFEST['languages']}"
    assert not processed_changed, f"processed content differs from the manifest: {processed_changed}"
    print("processed content matches the manifest")
elif MANIFEST is not None:
    print("processed files changed since the manifest:", processed_changed or "none")
pd.DataFrame(PROCESSED).T

,rows,content_sha256
cmsb_train,9543,6acddd6edd08c9b54d1d91cd303af427281f620c1818a7...
edos_train,14000,a0be07e192676f48d01e982f50ff46bc098bfb91e88a8e...
exist_train,3260,44de3fc289499a7d9b27f6f0fba08e9d28afce8159b1e3...
cmsb_dev,1363,af75e8510c4f31d4eacf3a9eed596f69b4bb06ebccfa04...
edos_dev,2000,2dda93e3d8cb8f544a05fc445ebc17b45d846b2afd743e...
exist_dev,489,1120f88ca336be3be8135c5015d3b0eba611271ba13aa4...
cmsb_test,2725,92d104d32b7a7785cb37761a223a6147d1bcb661ae6853...
edos_test,4000,364f28a20288aac7d7c44f8d86c367ab1a0aae4a2a1761...
exist_test,978,a882f967107ef5bb0a8f3eeb13450690c6b81ea741ed81...


In [15]:
OUT_DIR = DATA_ROOT / CONFIG["processed_dir"]


def out_path(name):
    # One folder per split; only 99_final_eval reads test/
    return OUT_DIR / name.rsplit("_", 1)[1] / f"{name}.parquet"


for s in ["train", "dev", "test"]:
    (OUT_DIR / s).mkdir(parents=True, exist_ok=True)
# Remove files from the older flat layout (by exact name only)
for name in TABLES:
    (OUT_DIR / f"{name}.parquet").unlink(missing_ok=True)
written = []
for name, (table, src) in TABLES.items():
    path = out_path(name)
    if write_parquet(table, path, src):
        written.append(name)
    # Round trip: what is on disk must hash to what was built
    assert content_hash(pq.read_table(path)) == PROCESSED[name]["content_sha256"], name
print("written:", written or "none (all unchanged)")

# Manifest and run record are written together, so a later crash can't leave one without the other
RUN_DIR = OUTPUT_ROOT / CONFIG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "config.json").write_text(json.dumps({**CONFIG, "env": ENV, "notebook_sha256": NOTEBOOK_SHA}, indent=2) + "\n")
(RUN_DIR / "pip_freeze.txt").write_text(PKG_FREEZE)
if not CHECK_MANIFEST:
    MANIFEST = {
        "written_by": CONFIG["run_name"],
        "env": ENV,
        "languages": CONFIG["languages"],
        "raw": {k: {"path": p, "bytes": (RAW_ROOT / p).stat().st_size, "sha256": RAW_SHA[k]} for k, p in RAW.items()},
        "processed": PROCESSED,
    }
    MANIFEST_PATH.parent.mkdir(exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2, sort_keys=True) + "\n")
    print("wrote", MANIFEST_PATH)
_ = (RUN_DIR / "data_manifest.json").write_text(json.dumps(MANIFEST, indent=2, sort_keys=True) + "\n")

written: ['cmsb_train', 'edos_train', 'exist_train', 'cmsb_dev', 'edos_dev', 'exist_dev', 'cmsb_test', 'edos_test', 'exist_test']
wrote /Users/pa1/Documents/Sexism-research/results/data_manifest.json


## Summary
Counts only; no post text. Train/dev show per-task class counts; test files show row counts and text flags only, so no test label statistics are seen here.

In [16]:
def summarise(name):
    df = pd.read_parquet(out_path(name))
    stats = {"rows": len(df), "empty_norm": int(df.empty_norm.sum()), "dup_rows": int(df.dup_group.notna().sum()),
             "leak_rows": int(df.leak_group.notna().sum())}
    if name.endswith("_test"):
        return stats
    stats["sexist_rate"] = round(df.sexist.mean(), 4) if df.sexist.notna().any() else None
    if "annotated" in df:
        stats["annotated"] = int(df.annotated.sum())
        stats["annotated_sexist_rate"] = round(df.sexist[df.annotated].mean(), 4)
    # Class counts follow each dataset's own task type
    for task, t in CONFIG["tasks"][name.split("_")[0]].items():
        col = df[t["column"]]
        if t["type"] == "multiclass":
            stats |= {f"{task}:{c}": int(col.eq(c).sum()) for c in t["classes"]}
        elif t["type"] == "multilabel":
            hard = col.dropna()
            stats |= {f"{task}:hard_labelled": len(hard), f"{task}:multi_label": int(hard.map(len).gt(1).sum())}
            stats |= {f"{task}:{c}": int(hard.map(lambda l: c in l).sum()) for c in t["classes"]}
    return stats


STATS = {name: summarise(name) for name in TABLES}
for ds in ["cmsb", "edos", "exist"]:
    display(pd.DataFrame({n: s for n, s in STATS.items() if n.startswith(ds)}).T.dropna(axis=1, how="all"))

,rows,empty_norm,dup_rows,leak_rows,sexist_rate,annotated,annotated_sexist_rate
cmsb_train,9543.0,67.0,543.0,108.0,0.1328,3954.0,0.3204
cmsb_dev,1363.0,12.0,76.0,19.0,0.1328,566.0,0.3198
cmsb_test,2725.0,16.0,158.0,33.0,NaN,NaN,NaN


,rows,empty_norm,dup_rows,leak_rows,sexist_rate,task_b:threats,task_b:derogation,task_b:animosity,task_b:prejudiced_discussions,task_c:1.1,task_c:1.2,task_c:2.1,task_c:2.2,task_c:2.3,task_c:3.1,task_c:3.2,task_c:3.3,task_c:3.4,task_c:4.1,task_c:4.2
edos_train,14000.0,0.0,0.0,0.0,0.2427,310.0,1590.0,1165.0,333.0,56.0,254.0,717.0,673.0,200.0,637.0,417.0,64.0,47.0,75.0,258.0
edos_dev,2000.0,0.0,0.0,0.0,0.2430,44.0,227.0,167.0,48.0,8.0,36.0,102.0,96.0,29.0,91.0,60.0,9.0,7.0,11.0,37.0
edos_test,4000.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rows,empty_norm,dup_rows,leak_rows,sexist_rate,task_1_3:hard_labelled,task_1_3:multi_label,task_1_3:IDEOLOGICAL-INEQUALITY,task_1_3:STEREOTYPING-DOMINANCE,task_1_3:OBJECTIFICATION,task_1_3:SEXUAL-VIOLENCE,task_1_3:MISOGYNY-NON-SEXUAL-VIOLENCE
exist_train,3260.0,0.0,4.0,2.0,0.3962,2864.0,766.0,481.0,613.0,492.0,316.0,304.0
exist_dev,489.0,0.0,0.0,0.0,0.4369,444.0,140.0,95.0,105.0,95.0,49.0,68.0
exist_test,978.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Run record, per the repo rules
(RUN_DIR / "metrics.json").write_text(
    json.dumps({"processed": PROCESSED, "written": written, "audit": AUDIT, "stats": STATS}, indent=2, default=str) + "\n"
)
(RUN_DIR / "README.md").write_text(f"""# {CONFIG['run_name']}
- Purpose: build processed train/dev/test Parquet files for CMSB, EDOS and EXIST 2025 (languages: {', '.join(CONFIG['languages'])}).
- Notebook: `notebooks/01_data_prep.ipynb` (sha256 `{NOTEBOOK_SHA[:12]}`); config `configs/data_prep.yaml` (sha256 `{CONFIG['config_sha256'][:12]}`).
- Mode: {'checked against' if CHECK_MANIFEST else 'rewrote'} `{CONFIG['manifest']}`.
- Outcome: {len(written)} of {len(TABLES)} processed files written ({', '.join(written) or 'none, all unchanged'}); hashes, audit flags and train/dev stats in `metrics.json`.
""")
print("saved to", RUN_DIR)

saved to /Users/pa1/Documents/Sexism-research/outputs/20261019-1557_all_prep_none_dataprep_s42


## Findings
See `docs/experiment-log.md` for the dated record of each run. As of the per-dataset task rebuild (2026-10-19):
- All raw-file, split-property and gold-consistency checks pass; EDOS Task A/B/C labels are consistent in both the aggregated and the individual files.
- Text flags are kept, not acted on: see the audit table above for empty, duplicate and cross-split inputs per dataset.
- Blocking (2026-10-19, after those runs): `data/splits/cmsb_split_s42.csv` and `data/processed/` are missing, so this notebook can't run until the split file is restored. Don't regenerate the split without a decision.
- Pending rebuild: the glued-`RT` fix (2 CMSB `text_norm` values) and the EXIST `n_annot` / `labels_n_annot` change are not in the current manifest. Once the split is restored, run once with `update_manifest: true`.
- Open: which CMSB flagged rows to exclude, and how EDOS Tasks B/C are evaluated (TODO in `configs/data_prep.yaml`).